# 내 EDA

## 0. 설정
데이터 로딩과 한글 폰트. `df` = train, `test` = test.

In [ ]:
import os, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}.get(platform.system(), "NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

DATA = "../data" if os.path.exists("../data") else "data"
df = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
print(df.shape, test.shape)

## 1. 호우사상 길이

- **사상**: 전체 시각 축에서 1시간 넘게 비면 새 사상 (AGENT_REFERENCE 7절)
- **길이**: 사상의 끝 시각 − 첫 시각 + 1h
- `event_table`은 사상마다 길이와 행 수를 돌려준다. 원본은 복사해서 쓴다.

In [ ]:
def event_table(frame):
    data = frame.copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"])
    ts = pd.Series(sorted(data["timestamp"].unique()))
    data["event_id"] = data["timestamp"].map(dict(zip(ts, (ts.diff() > pd.Timedelta("1h")).cumsum())))
    ev = data.groupby("event_id").agg(start=("timestamp", "min"), end=("timestamp", "max"), n_rows=("row_id", "size"))
    ev["length_h"] = (ev["end"] - ev["start"]) / pd.Timedelta("1h") + 1
    return data, ev

data, ev_tr = event_table(df)
_, ev_te = event_table(test)
pd.DataFrame({"train": ev_tr["length_h"].describe(), "test": ev_te["length_h"].describe()}).round(1)

### 분포
왼쪽은 사상 개수 기준 히스토그램(로그 축), 오른쪽은 누적분포. 점선은 타깃 창 6h.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
bins = np.logspace(0, np.log10(160), 25)
for name, ev, c in [("train", ev_tr, "#2F6DB5"), ("test", ev_te, "#E07B39")]:
    L = ev["length_h"]
    axes[0].hist(L, bins=bins, alpha=.55, color=c, edgecolor="white", label=f"{name} ({len(L)}개, 중앙 {L.median():.0f}h)")
    axes[1].step(np.sort(L), np.arange(1, len(L) + 1) / len(L), where="post", color=c, lw=2, label=name)
for ax, t, yl in zip(axes, ["사상 길이 분포", "사상 길이 누적분포"], ["사상 수", "누적 비율"]):
    ax.set(xscale="log", title=t, xlabel="사상 길이 (시간, 로그)", ylabel=yl)
    ax.axvline(6, color="gray", ls=":")
    ax.legend()
plt.tight_layout()
plt.show()

### 타깃과의 관계 (전체 vs 비조석)
사상 길이 구간별 타깃 평균과 p95. 비조석은 조석 15곳과 고정 ST0027을 뺀 것.

⚠️ 사상 길이는 사상이 끝나야 아는 값이라 **모델 변수로 쓰면 누수**다. 분석용으로만 본다.

In [ ]:
reg = pd.read_csv("cache/results/nb07_regimes.csv")
excl = set(reg.loc[reg["group"].isin(["조석", "고정"]), "station_id"])

d = data[["event_id", "station_id", "target_maxrise_6h"]].copy()
d["구간"] = pd.cut(d["event_id"].map(ev_tr["length_h"]), [0, 3, 6, 12, 24, 48, 200],
                  labels=["1-3h", "4-6h", "7-12h", "13-24h", "25-48h", "49h+"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, (name, g) in zip(axes, [("전체", d), ("비조석", d[~d["station_id"].isin(excl)])]):
    s = g.groupby("구간", observed=True)["target_maxrise_6h"]
    pd.DataFrame({"평균": s.mean(), "p95": s.quantile(.95)}).plot.bar(ax=ax, rot=0, color=["#2F6DB5", "#8FB3DE"])
    ax.set(title=name, xlabel="사상 길이 구간", ylabel="target_maxrise_6h (m)")
plt.tight_layout()
plt.show()

### 해석
- 사상 길이는 오른쪽 꼬리가 길어. train 중앙값 9h, 최대 155h이고 test는 중앙값 10h, 최대 62h야.
- train과 test의 누적분포는 거의 겹쳐. test에는 초장기 사상이 없다는 점만 달라.
- 6h 이하 사상이 개수로는 40%인데 행으로는 9%야. 가장 긴 사상 1개가 train 행의 약 10%라서, 사상 단위로 fold를 나누면 크기가 치우칠 수 있어.
- 사상 길이와 타깃의 스피어만은 0.02로 거의 0이야. 그런데 25h 넘는 사상에서 p95가 크게 뛰어. 큰 상승이 긴 사상에 몰려 있다는 뜻이야.
- 이 EDA는 전체 데이터 기준이야. 모델링할 땐 train 기준으로 다시 계산해.

## 2. 두 강우 소스 비교: 한쪽만 비가 올 때

- IMERG(위성)와 지상강수를 창(1h / 6h / 24h)마다 `> 0`이면 "비 옴"으로 보고 네 칸으로 나눈다: **둘 다 / IMERG만 / 지상만 / 둘 다 없음**
- 막대는 행 비율이다.

In [ ]:
CATS = ["둘 다", "IMERG만", "지상만", "둘 다 없음"]
COLORS = ["#2F6DB5", "#8FB3DE", "#E07B39", "#D0D0D0"]

def rain_cat(frame, w):
    i, s = frame[f"imerg_rain_{w}"] > 0, frame[f"sfc_rain_{w}"] > 0
    return pd.Series(np.select([i & s, i & ~s, ~i & s], CATS[:3], CATS[3]), index=frame.index)

share = pd.DataFrame({w: rain_cat(df, w).value_counts(normalize=True) for w in ["1h", "6h", "24h"]}).T[CATS]

ax = share.plot.barh(stacked=True, color=COLORS, figsize=(12, 3), edgecolor="white")
for c in ax.containers:
    ax.bar_label(c, labels=[f"{v:.0%}" if v > .03 else "" for v in c.datavalues], label_type="center", fontsize=9)
ax.set(title="창별 두 강우 소스의 비 옴/안 옴 조합 (train 전체 행)", xlabel="행 비율", ylabel="강우 누적 창")
ax.invert_yaxis()
ax.legend(ncol=4, bbox_to_anchor=(0.5, -0.35), loc="upper center")
plt.tight_layout()
plt.show()

## 결론 : 1. IMERG가 지상에 비해 훨씬 더 잘 강수를 잡는다.

### 한쪽만 올 때 얼마나 오고, 수위는 오르나 (6h 창)

- 1~2번: 한쪽만 비가 올 때, 비를 본 쪽의 강우량 분포. 1번 log1p 축, 2번 mm 그대로 (0.5mm 칸)
- 3~4번: 조합별 타깃 평균과 p95. 전체 vs 비조석(조석 15곳 + ST0027 제외)

In [ ]:
d = df[["station_id", "imerg_rain_6h", "sfc_rain_6h", "target_maxrise_6h"]].assign(조합=rain_cat(df, "6h"))
reg = pd.read_csv("cache/results/nb07_regimes.csv")
excl = set(reg.loc[reg["group"].isin(["조석", "고정"]), "station_id"])

fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
one_side = [("IMERG만", "imerg_rain_6h", COLORS[1], "IMERG만 → IMERG 강우"), ("지상만", "sfc_rain_6h", COLORS[2], "지상만 → 지상 강우")]
for cat, col, c, lab in one_side:
    x = d.loc[d["조합"] == cat, col]
    axes[0].hist(np.log1p(x), bins=np.linspace(0, np.log1p(60), 30), density=True, alpha=.6, color=c, label=lab)
    axes[1].hist(x.clip(upper=25), bins=np.arange(0, 25.5, 0.5), density=True, alpha=.6, color=c, label=f"{lab} (중앙 {x.median():.1f}mm)")
axes[0].set(title="한쪽만 비 올 때 그쪽 강우량 (log1p 축)", xlabel="log1p(6h 강우, mm)", ylabel="밀도")
axes[1].set(title="같은 분포 (mm, 25mm 이상은 끝 칸)", xlabel="6h 강우 (mm)", ylabel="밀도")
axes[0].legend()
axes[1].legend()

for ax, (name, g) in zip(axes[2:], [("전체", d), ("비조석", d[~d["station_id"].isin(excl)])]):
    s = g.groupby("조합")["target_maxrise_6h"]
    pd.DataFrame({"평균": s.mean(), "p95": s.quantile(.95)}).loc[CATS].plot.bar(ax=ax, rot=0, color=["#2F6DB5", "#8FB3DE"])
    ax.set(title=f"{name}: 6h 강우 조합별 타깃", xlabel="", ylabel="target_maxrise_6h (m)")
axes[3].sharey(axes[2])
plt.tight_layout()
plt.show()

## IMERG 단독으로 강수를 포찰하면 거의 1mm 수준의 강수다.

### 해석
- 한쪽만 비가 오는 경우가 꽤 많아. 1h 창에서는 IMERG만 35%, 지상만 4%야. 창이 길어질수록 "둘 다"가 늘어나서 24h에서는 66%가 돼.
- 어긋나는 쪽은 대부분 IMERG만 비를 보는 경우야. IMERG가 약한 비를 잘 잡고 지상은 약한 비를 놓친다는 기존 사실과 맞아.
- 한쪽만 비가 올 때 그 양은 6h 중앙값 약 1.1mm로 약한 비야. 그래도 상위 1%는 20mm를 넘어서, 한쪽이 큰 비를 통째로 놓친 경우도 있어.
- 수위는 둘 다 비를 볼 때만 확실히 올라. 비조석 기준 p95가 둘 다 0.72m, 지상만 0.26m, IMERG만 0.15m, 둘 다 없음 0.11m야.
- 지상만 비를 본 경우가 IMERG만 본 경우보다 더 오르는 건, 지상 관측이 관측소 근처 비를 잡기 때문일 수 있어. (추정이고 확인은 안 했어)
- 이 EDA는 전체 데이터 기준이야. 모델링할 땐 train 기준으로 다시 계산해.

## 3. 레이더와 짝 비교: 레이더×지상, 레이더×IMERG

- 레이더는 창이 없는 t 시점 값이라 **1h 창**과 맞춘다
- 레이더 "비 옴" = `radar_reflectivity_p95 ≥ 20` (보통 약한 비 기준 20 dBZ. 단위가 미기재라 dBZ라고 가정). `ge20_fraction > 0`은 행 72%가 걸려서 너무 느슨함
- 짝마다 **둘 다 / 레이더만 / 상대만 / 둘 다 없음** 네 칸. 윗줄 = 레이더×지상, 아랫줄 = 레이더×IMERG. 칸마다 행 비율, 그리고 값의 평균과 p95 (전체 vs 비조석)
- `plot_pairs(값, 라벨, w)`로 어떤 값이든 같은 틀에서 본다. `w`는 지상·IMERG 강우 창 (기본 1h)

In [ ]:
radar = df["radar_reflectivity_p95"] >= 20

def plot_pairs(y, label, w="1h"):
    pairs = {"레이더×지상": ("지상", df[f"sfc_rain_{w}"] > 0), "레이더×IMERG": ("IMERG", df[f"imerg_rain_{w}"] > 0)}
    fig, axes = plt.subplots(2, 3, figsize=(16, 7))
    for row, (title, (other, x)) in zip(axes, pairs.items()):
        cats = ["둘 다", "레이더만", f"{other}만", "둘 다 없음"]
        d = df[["station_id"]].assign(값=y, 조합=np.select([radar & x, radar & ~x, ~radar & x], cats[:3], cats[3]))

        share = d["조합"].value_counts(normalize=True).reindex(cats)
        row[0].bar(cats, share, color=COLORS)
        for k, v in enumerate(share):
            row[0].text(k, v, f"{v:.0%}", ha="center", va="bottom", fontsize=9)
        row[0].set(title=f"{title}: 행 비율", ylabel="행 비율")

        for ax, (name, g) in zip(row[1:], [("전체", d), ("비조석", d[~d["station_id"].isin(excl)])]):
            s = g.groupby("조합")["값"]
            pd.DataFrame({"평균": s.mean(), "p95": s.quantile(.95)}).reindex(cats).plot.bar(ax=ax, rot=0, color=["#2F6DB5", "#8FB3DE"])
            ax.set(title=f"{title} · {name}", xlabel="", ylabel=label)
    axes[0, 2].sharey(axes[0, 1]); axes[1, 1].sharey(axes[0, 1]); axes[1, 2].sharey(axes[0, 1])
    fig.suptitle(label, y=1.01)
    plt.tight_layout()
    plt.show()

### 3-1. 타깃: 앞으로 6h 최대 상승량

In [ ]:
plot_pairs(df["target_maxrise_6h"], "target_maxrise_6h (m)")

## 결론 : 레이더가 지상보다는 많이 잡는데, imerg보다는 못잡는다. 

### 해석 (타깃)
- **레이더×지상**: 어긋나는 쪽은 주로 레이더만 비를 보는 경우(25%)야. 지상은 레이더가 잡는 비를 많이 놓쳐.
- **레이더×IMERG**: 둘 다 비를 보는 경우가 47%로 레이더×지상(32%)보다 훨씬 많아. 레이더는 지상보다 IMERG랑 더 잘 맞아.
- 두 짝 모두 둘 다 비를 볼 때만 확실히 올라. 비조석 p95는 레이더×지상 0.89m, 레이더×IMERG 0.74m야. 레이더와 지상이 같이 본 비가 더 센 비라는 뜻이야.
- 한쪽만 볼 때는 레이더만 0.30m, 지상만 0.36m, IMERG만 0.19m야. 한 소스만 봤을 때 IMERG만 본 비가 제일 약해. IMERG가 약한 비까지 잡는다는 기존 사실과 맞아.

### 3-2. 지난 1h 수위 증가량: `dwl_1h = wl_t − wl_t_minus_1h`
강우 1h 창과 같은 시간대의 실제 수위 변화. t까지의 값이라 누수 아님.

In [ ]:
plot_pairs(df["wl_t"] - df["wl_t_minus_1h"], "지난 1h 수위 증가량 dwl_1h (m)")

## 앞으로 한시간 수위 증가로 볼때, 지상에서 못잡은 자료라해도 별 의미 없는것 같다. 

### 해석 (수위 증가량)
- 값 크기가 타깃보다 한 자릿수 작아. 1h 동안의 변화라서 그래. 둘 다 비를 볼 때 비조석 p95는 레이더×지상 0.15m, 레이더×IMERG 0.11m야.
- 타깃이랑 다른 점이 있어. **지상만 비를 볼 때 이미 수위가 올라** (p95 0.10m). 레이더만(0.04~0.06m)이나 IMERG만(0.03m)은 둘 다 없음(0.03m)과 크게 다르지 않아.
- 지상 강우는 관측소 근처 비라서 바로 수위에 반응하고, 레이더·IMERG는 넓은 영역을 보니까 반응이 늦게 나타나는 걸로 보여. (추정)
- 레이더만 볼 때 수위는 지금 거의 안 오르는데(0.04~0.06m) 타깃은 0.30m야. 앞으로 오를 비를 레이더가 먼저 잡고 있을 수 있어.
- 조석 관측소를 빼도 값이 거의 안 변해. 조석은 1h 변화보다 6h 최대 상승에 더 크게 끼어들어.
- 이 EDA는 전체 데이터 기준이야. 모델링할 땐 train 기준으로 다시 계산해.

### 3-2b. 창을 맞춘 수위 증가량: 6h 강우 × `wl_t − wl_t_minus_6h`, 24h 강우 × `wl_t − wl_t_minus_24h`
3-2와 같은 틀인데, 지상·IMERG "비 옴"을 **수위 변화와 같은 창**(6h 누적, 24h 누적 > 0)으로 나눈다. 강우 창이 1h·6h·24h뿐이라 12h 대신 24h로 본다.

⚠️ 레이더는 창이 없는 t 시점 값이라 레이더만 시간 폭이 안 맞는다. 레이더 조합은 "지금 레이더에 비가 걸리나"로 읽는다.

In [ ]:
plot_pairs(df["wl_t"] - df["wl_t_minus_6h"], "6h 강우 × 지난 6h 수위 증가량 (m)", w="6h")

In [ ]:
plot_pairs(df["wl_t"] - df["wl_t_minus_24h"], "24h 강우 × 지난 24h 수위 증가량 (m)", w="24h")

### 해석 (창을 맞춘 수위 증가량)
- 창을 늘리면 지상·IMERG가 비를 본 행이 늘어나. 레이더×IMERG 둘 다는 6h 54%, 24h 55%이고, 레이더만은 6h 3%, 24h 1%로 거의 사라져. 레이더가 잡는 비는 긴 창의 IMERG에 대부분 걸려.
- **둘 다 비를 볼 때가 제일 크게 올라.** 비조석 p95가 레이더×지상 6h 0.58m, 24h 0.96m이고 레이더×IMERG 6h 0.46m, 24h 0.83m야.
- **지상만은 둘 다의 80% 정도야** (6h 0.46m, 24h 0.78m). 지상 강우는 창을 맞춰도 수위 증가를 잘 따라가.
- **IMERG만은 절반 정도야** (6h 0.21m, 24h 0.46m). 1h 창에서는 둘 다 없음과 비슷했는데, 창을 길게 맞추니 분명히 올라.
- **레이더만은 둘 다 없음과 같아** (6h 0.08m vs 0.07m, 24h 0.10m vs 0.08m). 평균은 오히려 음수(−0.01~−0.10m)로 수위가 내려가는 중이야. 지난 6·24시간 동안 지상·IMERG 모두 비가 없는데 지금 레이더에만 걸리는 경우라서, 비가 막 시작됐거나 레이더 잡음일 수 있어. (추정)
- 창을 안 맞췄을 때 레이더만이 커 보였던 건(12h p95 0.57m), 앞서 온 비를 1h 창 IMERG가 놓쳐서 레이더만 칸에 섞였기 때문이야.
- 24h 레이더×IMERG의 레이더만은 행 0.8%라 수치가 흔들릴 수 있어.
- 조석 관측소를 빼도 순서는 그대로야.
- 이 EDA는 전체 데이터 기준이야. 모델링할 땐 train 기준으로 다시 계산해.

## 4. 레이더+IMERG만 비를 볼 때, 그 뒤에 무슨 일이

- **레이더+IMERG만** = 1h 창에서 레이더(p95 ≥ 20)와 IMERG는 비, 지상은 0. 비교용으로 셋 다 / IMERG만 / 레이더만 / 셋 다 없음도 같이 그린다
- 같은 관측소·같은 사상 안에서 k시간 뒤 행을 찾아 **`WL(t+k) − WL(t)`** (k = 0~12)와 **k시간 뒤 지상 강우가 잡힐 확률**을 본다
- t 이후 값을 쓰는 분석이라 **모델 변수로 쓰면 누수**. 분석용으로만
- 사상이 짧으면 뒤 행이 없어서 빠진다 (6h 뒤는 행 64%, 12h 뒤는 43%만 남음). 멀리 갈수록 긴 사상 위주로 남는다는 점을 감안해서 봐

In [ ]:
I, S, R = data["imerg_rain_1h"] > 0, data["sfc_rain_1h"] > 0, data["radar_reflectivity_p95"] >= 20
GROUPS = {"레이더+IMERG만": R & I & ~S, "셋 다": R & I & S, "IMERG만": I & ~R & ~S,
          "레이더만": R & ~I & ~S, "셋 다 없음": ~R & ~I & ~S}
STYLE = {"레이더+IMERG만": ("#E07B39", 3), "셋 다": ("#2F6DB5", 1.8), "IMERG만": ("#8FB3DE", 1.8),
         "레이더만": ("#7A9A5A", 1.8), "셋 다 없음": ("#9A9A9A", 1.8)}
K = range(13)

look = data.set_index(["station_id", "event_id", "timestamp"])[["wl_t", "sfc_rain_1h"]]
fut_wl, fut_rain = {}, {}
for k in K:
    f = look.reindex(pd.MultiIndex.from_arrays([data["station_id"], data["event_id"], data["timestamp"] + pd.Timedelta(hours=k)]))
    fut_wl[k] = f["wl_t"].values - data["wl_t"].values
    fut_rain[k] = np.where(f["wl_t"].isna(), np.nan, f["sfc_rain_1h"].values > 0)
fut_wl, fut_rain = pd.DataFrame(fut_wl, index=data.index), pd.DataFrame(fut_rain, index=data.index)
nontidal = ~data["station_id"].isin(excl)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name, m in GROUPS.items():
    c, lw = STYLE[name]
    axes[0].plot(K, fut_wl[m].mean(), color=c, lw=lw, marker="o", ms=3, label=f"{name} ({m.mean():.0%})")
    axes[1].plot(K, fut_wl[m & nontidal].mean(), color=c, lw=lw, marker="o", ms=3, label=name)
    axes[2].plot(K, fut_rain[m].mean(), color=c, lw=lw, marker="o", ms=3, label=name)
for ax, t in zip(axes[:2], ["전체", "비조석"]):
    ax.axhline(0, color="black", lw=.8)
    ax.set(title=f"{t}: k시간 뒤 수위 변화 평균", xlabel="t 이후 시간 k (h)", ylabel="WL(t+k) - WL(t) (m)")
axes[1].sharey(axes[0])
axes[2].set(title="k시간 뒤 지상 강우가 잡힐 확률", xlabel="t 이후 시간 k (h)", ylabel="P(sfc_rain_1h > 0)", ylim=(0, 1))
axes[0].legend(fontsize=9)
plt.tight_layout()
plt.show()

## 전체 기간으로 볼때, 레이더만 강수를 잡은건 의미가 없고, imerg가 잡은건 6시간후, 레이더와 imerg가 잡은건 3시간 후 부터 영향이 있는것으로 보임.

### 해석
- 레이더+IMERG만 비를 볼 때는 **처음엔 수위가 거의 안 움직이다가 천천히 올라**. 평균이 1h 뒤 +0.001m, 3h 뒤 +0.01m, 6h 뒤 +0.04m, 12h 뒤 +0.11m야.
- 셋 다 비를 볼 때(6h 뒤 +0.18m, 12h 뒤 +0.31m)보다는 한참 작아. 그래도 IMERG만(6h 뒤 +0.004m)이나 레이더만(−0.015m)과 비교하면 분명히 올라.
- 지상 강우도 **뒤늦게 따라와**. 지금은 0인데 1h 뒤 34%, 3h 뒤 44%, 6h 뒤 48% 확률로 지상에서도 비가 잡혀. IMERG만(6h 뒤 35%)이나 레이더만(27%)보다 높아.
- 정리하면 레이더와 IMERG가 같이 잡았는데 지상은 아직 0인 경우는 **비가 관측소 쪽으로 오는 중**일 가능성이 커. (추정)
- 중앙값은 12h 뒤에도 +0.02m야. 평균이 오르는 건 일부 행이 크게 올라서 그래.
- 전체와 비조석이 거의 같아. 이 조합에는 조석 행이 적어.
- 이 EDA는 전체 데이터 기준이야. 모델링할 땐 train 기준으로 다시 계산해.

## 5. 유역환경 주성분(PC1·PC2) × 강우 창(1h / 6h / 24h)

- **질문**: 유역환경 주성분에 따라 1h·6h·24h 강우와 수위 상승의 관계가 달라지나
- **주성분**: 관측소 465곳의 유역환경 7개(면적·표고·차수는 log) 표준화 → PCA (report_final 5.2와 같은 방식)
- **3분위**: 주성분 점수 순위로 관측소를 하/중/상으로 나눈다 (관측소 단위)
- **강우**: 지상강수 `sfc_rain_1h/6h/24h`. 수위 상승 = 타깃 `target_maxrise_6h`
- **반응 강도** = 비가 온 행(창 강우 > 0)에서 강우와 타깃의 스피어만 ρ. 모두 t까지 값이라 누수 없음

In [ ]:
ENV = ["cat_area", "wsarea", "mean_elevation", "mean_slope", "urban_ratio", "forest_ratio", "stream_order"]
st = df.groupby("station_id")[ENV].first().astype(float)
X = st.copy()
for c in ["cat_area", "wsarea", "mean_elevation", "stream_order"]:
    X[c] = np.log(X[c])
Z = (X - X.mean()) / X.std()
U, S, Vt = np.linalg.svd(Z.to_numpy(), full_matrices=False)
ev = S**2 / (S**2).sum()
PCS = ["PC1", "PC2", "PC3"]
sc = pd.DataFrame((U * S)[:, :3], index=st.index, columns=PCS)
pd.DataFrame(Vt[:3].T, index=ENV, columns=[f"{p} ({e:.0%})" for p, e in zip(PCS, ev)]).round(2)

### 적재값 vs 점수와의 상관
적재값은 축의 **방향**만 말해 준다. 변수가 실제로 어느 축에 담겼는지는 **변수와 주성분 점수의 상관**으로 본다. 3분위는 순위로 나누니까 스피어만도 같이 본다.

In [ ]:
both = pd.concat([X, sc], axis=1)
pd.concat({"피어슨": both.corr().loc[ENV, PCS], "스피어만": both.corr(method="spearman").loc[ENV, PCS]}, axis=1).round(2)

- 도시화율은 적재값이 PC3에서 제일 크지만(0.49), 점수와의 상관은 **PC1이 0.68(순위 0.72)**로 가장 커. PC3와는 순위 상관이 0.03이라 PC3 3분위는 도시화와 거의 무관해.
- PC1은 분산이 PC3의 3.4배라서 같은 적재값이라도 담는 양이 훨씬 많아.
- 그래서 아래에서는 **PC1(산지 ↔ 저지·도시)**, **PC2(하천 규모)** 두 축만 본다.

### PC1 × PC2 관측소 지도
점 = 관측소. 왼쪽은 도시화율, 오른쪽은 상류면적(log) 색. 화살표는 변수 방향, 점선은 3분위 경계.

In [ ]:
reg = pd.read_csv("cache/results/nb07_regimes.csv").set_index("station_id")["group"]
excl = set(reg[reg.isin(["조석", "고정"])].index)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
k = np.abs(sc[["PC1", "PC2"]]).max().max() / np.abs(Vt[:2]).max() * .8
for ax, (col, lab) in zip(axes, [("urban_ratio", "도시화율"), ("wsarea", "log 상류면적")]):
    m = ax.scatter(sc["PC1"], sc["PC2"], c=X[col], cmap="viridis", s=14, alpha=.8)
    tid = sc.index.isin(excl)
    ax.scatter(sc.loc[tid, "PC1"], sc.loc[tid, "PC2"], facecolors="none", edgecolors="#E07B39", s=40, label="조석·고정")
    for p, f in [("PC1", ax.axvline), ("PC2", ax.axhline)]:
        for q in sc[p].quantile([1/3, 2/3]):
            f(q, color="gray", ls=":", lw=1)
    for i, c in enumerate(ENV):
        ax.arrow(0, 0, Vt[0, i] * k, Vt[1, i] * k, color="k", width=.01, head_width=.12)
        ax.text(Vt[0, i] * k * 1.12, Vt[1, i] * k * 1.12, c, fontsize=8)
    fig.colorbar(m, ax=ax, label=lab)
    ax.set(xlabel=f"PC1 ({ev[0]:.0%}) 산지 ← → 저지·도시", ylabel=f"PC2 ({ev[1]:.0%}) 하천 규모", title=f"색 = {lab}")
axes[0].legend(loc="lower left")
plt.tight_layout(); plt.show()

### 강우량 구간별 상위 − 하위 평균 타깃 (비조석)
비가 온 행만 창마다 강우량 8분위(1 = 가장 약한 비, 8 = 가장 센 비)로 나눈다. 분위마다 **3분위 상위 관측소 평균 타깃 − 하위 관측소 평균 타깃**을 찍는다.
- 0보다 위 = 상위(PC1은 저지·도시, PC2는 큰 하천)가 더 오름, 아래 = 하위(산지, 작은 하천)가 더 오름
- 창마다 강우량 단위가 달라서 x는 mm가 아니라 분위로 맞췄다

In [ ]:
WIN, PC2S = ["1h", "6h", "24h"], ["PC1", "PC2"]
p5 = df[["station_id", "target_maxrise_6h"] + [f"sfc_rain_{w}" for w in WIN]].copy()
for p in PC2S:
    p5[p] = p5["station_id"].map(pd.qcut(sc[p].rank(method="first"), 3, labels=["하", "중", "상"]))
nt5 = p5[~p5["station_id"].isin(excl)]

WCOL = {"1h": "#8FB3DE", "6h": "#2F6DB5", "24h": "#E07B39"}
NAME = {"PC1": "PC1: 저지·도시 - 산지", "PC2": "PC2: 큰 하천 - 작은 하천"}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, p in zip(axes, PC2S):
    for w in WIN:
        x = nt5[nt5[f"sfc_rain_{w}"] > 0].copy()
        x["분위"] = pd.qcut(x[f"sfc_rain_{w}"].rank(method="first"), 8, labels=range(1, 9))
        m = x.groupby(["분위", p], observed=True)["target_maxrise_6h"].mean().unstack()
        ax.plot(m.index.astype(int), m["상"] - m["하"], "-o", color=WCOL[w], lw=2.2, ms=5, label=f"{w} 강우")
    ax.axhline(0, color="gray", lw=1)
    ax.set(title=NAME[p], xlabel="강우량 8분위 (1 약 → 8 강)", ylabel="상위 - 하위 평균 타깃 (m)", xticks=range(1, 9))
axes[0].legend()
plt.tight_layout(); plt.show()

### 반응 강도 ρ (전체 vs 비조석)
막대 = 3분위별 ρ(강우, 타깃). 창마다 하/중/상 순서가 어떻게 바뀌는지 본다.

In [ ]:
PAL = {"하": "#C8D8EC", "중": "#8FB3DE", "상": "#2F6DB5"}
def rho_table(g):
    out = {}
    for p in PC2S:
        for w in WIN:
            x = g[g[f"sfc_rain_{w}"] > 0]
            out[(p, w)] = x.groupby(p, observed=True)[[f"sfc_rain_{w}", "target_maxrise_6h"]].apply(lambda q: q[f"sfc_rain_{w}"].corr(q["target_maxrise_6h"], method="spearman"))
    return pd.DataFrame(out).T

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)
for row, (name, g) in zip(axes, [("전체", p5), ("비조석", nt5)]):
    r = rho_table(g)
    for ax, p in zip(row, PC2S):
        r.loc[p].plot.bar(ax=ax, rot=0, color=PAL, width=.8, legend=False)
        ax.set(title=f"{name} · {p}", xlabel="강우 창", ylabel="ρ(강우, 타깃)")
axes[0, 0].legend(title="3분위")
plt.tight_layout(); plt.show()
rho_table(nt5).round(2)

### 해석
- **지도**: 도시화율이 높은 관측소(노랑·초록)는 PC1 오른쪽 아래에 몰려 있고, 왼쪽(산지)은 거의 다 도시화 10% 미만이야. 상류면적은 PC2를 따라 위로 갈수록 커져. 두 축이 도시화와 규모를 거의 따로 나눠 줘.
- **PC1(산지 ↔ 저지·도시)**: 약한 비(1~6분위)에서는 도시가 산지보다 0.03~0.05m 더 올라. 그런데 센 비(8분위)에서는 뒤집혀서 6h −0.06m, 24h −0.08m로 산지가 더 올라. 1h 강우는 8분위에서도 차이가 0이야. ρ도 하/상 1h 0.42/0.34, 6h 0.42/0.27, 24h 0.28/0.14로 창이 길수록 산지 쪽 반응이 더 커.
- **PC2(하천 규모)**: 6h·24h 강우에서 1~6분위는 작은 하천이 0.02~0.05m 더 오르다가, 8분위에서 큰 하천이 **+0.20m** 더 올라. 1h 강우는 8분위에서도 +0.04m뿐이야. 24h ρ도 작은 하천 0.08, 큰 하천 0.30이야. 큰 유역은 긴 시간 모인 비가 뒤늦게 내려오는 걸로 보여. (추정)
- 정리하면 **짧은 비(1h)는 유역환경과 상관없이 비슷하게 반응**하고, **긴 비(6h·24h)가 셀 때 산지와 큰 하천이 더 크게 반응**해. 긴 창 강우 × PC1·PC2 상호작용을 변수로 넣어 볼 만해.
- 이 EDA는 전체 데이터 기준이야. 모델링할 땐 train 기준으로 다시 계산해.